# 01 — Exploratory Data Analysis: Heart Disease Dataset

This notebook performs a comprehensive EDA on the Cleveland Heart Disease dataset (303 rows, 14 columns).

**Key preprocessing:**
- Replace `'?'` values in `ca` and `thal` columns with `NaN`
- Binarize the target: values > 0 become 1 (presence of heart disease)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASETS, RAW_DATA_DIR, SEED
from src.data.loader import load_raw_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Data

In [ ]:
df = load_raw_dataset('heart')
print(f"Shape: {df.shape}")
df.head()

## 2. Clean Data — Handle '?' Values and Binarize Target

In [ ]:
# Replace '?' with NaN in ca and thal columns
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')

# Binarize target: 0 stays 0, 1-4 become 1
df['target'] = (df['target'] > 0).astype(int)

print("Unique target values after binarization:", df['target'].unique())
print(f"Missing values in 'ca': {df['ca'].isna().sum()}")
print(f"Missing values in 'thal': {df['thal'].isna().sum()}")

## 3. Basic Info and Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Missing Values Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap — Heart Disease Dataset')
plt.tight_layout()
plt.show()

## 5. Class Balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['target'].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_xlabel('Target (0 = No Disease, 1 = Disease)')
ax.set_ylabel('Count')
ax.set_title('Class Distribution — Heart Disease')
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Heart Disease Dataset')
plt.tight_layout()
plt.show()

## 7. Feature Distributions (KDE) Split by Target

In [ ]:
numeric_features = DATASETS['heart']['numeric_features']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    for label in [0, 1]:
        subset = df[df['target'] == label][col].dropna()
        sns.kdeplot(subset, ax=ax, label=f'Target={label}', fill=True, alpha=0.4)
    ax.set_title(f'{col} Distribution by Target')
    ax.legend()

# Hide unused subplot
for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('KDE Distributions of Numeric Features — Heart Disease', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Box Plots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    sns.boxplot(x='target', y=col, data=df, ax=ax)
    ax.set_title(f'{col} — Outliers')

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots of Numeric Features — Heart Disease', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Pair Plot for Top Correlated Features

In [ ]:
# Select the top 4 features most correlated with target
top_features = corr['target'].drop('target').abs().sort_values(ascending=False).head(4).index.tolist()
print(f"Top correlated features: {top_features}")

pair_df = df[top_features + ['target']].dropna()
g = sns.pairplot(pair_df, hue='target', diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.5})
g.figure.suptitle('Pair Plot — Top Correlated Features with Target', y=1.02)
plt.show()

## Summary

Key findings from this EDA:
- The dataset has 303 samples with a reasonably balanced binary target.
- Columns `ca` and `thal` contain missing values (originally encoded as `'?'`).
- Features like `thalach`, `oldpeak`, `cp`, and `exang` show strong correlation with the target.
- Some outliers exist in `chol`, `trestbps`, and `oldpeak` — addressed by the preprocessing pipeline.